# Capstone Part 3: Deep Learning, MLflow and Final Checks (Google Colab)

**What this notebook does, in order:**
1. Installs the extra packages the project needs (SHAP for explanations, MLflow for experiment records).
2. Clones this project's GitHub repository and moves into the Part 3 folder.
3. Runs the test suite that checks the project's rules hold (one row per hour, the outage window excluded, the chronological split, and more).
4. Runs the whole Part 3 pipeline, including the LSTM (deep learning) model.
5. Shows the deep-learning results and the MLflow experiment records.

**How to run:** click *Runtime → Run all*. No files need to be uploaded — everything comes from the public GitHub repository.

## Step 1. Install the extra packages

PyTorch is already installed in Colab. This installs SHAP (explains the deep-learning model) and MLflow (keeps experiment records).
It can take a minute or two.

In [ ]:
!pip install -q shap mlflow
import sys, torch
print("Python", sys.version.split()[0], "| PyTorch", torch.__version__)

## Step 2. Clone the project from GitHub

This downloads the repository and moves into the Part 3 folder. Nothing needs to be uploaded by hand.

In [ ]:
import os

!git clone --depth 1 https://github.com/SueLeeNUS/2026-nus-smart-city-traffic-capstone.git
%cd 2026-nus-smart-city-traffic-capstone/part3_machine_learning
print("Project files ready:", sorted(os.listdir(".")))

## Step 3. Run the tests

The tests check the rules the project depends on: one row per hour, the holiday count, the outage window being excluded,
and the chronological split. All should pass (16 tests).

In [ ]:
!python -m unittest discover -s tests 2>&1 | tail -5

## Step 4. Run the full pipeline

This trains every model, including the LSTM, and writes the results to `outputs/`.
It takes several minutes, mostly for the LSTM. The last lines should say *Pipeline finished*.

In [ ]:
!python run_all.py > run_log.txt 2>&1
!tail -n 15 run_log.txt

## Step 5. Check the deep-learning model

Shows the LSTM's test results (its traffic-volume error and its high-risk performance, when the project records them),
and which inputs mattered most according to the SHAP-style explanation.

In [ ]:
import pandas as pd

for name in ["lstm_metrics.csv", "lstm_surrogate_importance.csv"]:
    path = f"outputs/metrics/{name}"
    print("\n==", name)
    if os.path.exists(path):
        print(pd.read_csv(path).to_string(index=False))
    else:
        print("Not produced. Check run_log.txt for a message about deep learning.")

## Step 6. Check the MLflow records

MLflow keeps a list of every model run, with its settings and scores. The table below lists the runs.

In [ ]:
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")  # same fix as src/tracking.py, needed again here
                                                            # because this cell talks to MLflow directly,
                                                            # not through the project's own code

import mlflow
from src import config

mlflow.set_tracking_uri(config.TRACKING_DIR.as_uri())
runs = mlflow.search_runs(search_all_experiments=True)
if len(runs):
    cols = [c for c in ["experiment_id", "status", "tags.mlflow.runName"] if c in runs.columns]
    print(runs[cols].to_string())
else:
    print("No MLflow runs found. Check run_log.txt for a warning about MLflow.")